# 🥇 Pipeline End-to-End: Arquitetura Medalhão (Bronze ➔ Silver ➔ Gold)

Este notebook implementa a arquitetura completa de dados macroeconômicos e agropecuários:
1. **Bronze:** Ingestão da API do Banco Central (IPCA) e Cotações do Boi Gordo em tabelas Delta brutas com timestamps de auditoria.
2. **Silver:** Limpeza, conformidade de tipos numéricos e join temporal por competência mensal (`yyyy-MM-01`).
3. **Gold:** Lógica de negócio analítica com Window Functions (`lag`), cálculo de variações relativas e categorização de cenários.

In [ ]:
# ==========================================================
# 1. CONFIGURAÇÃO DO UNITY CATALOG
# ==========================================================
CATALOG = "workspace"
BRONZE = "bronze_economia"
SILVER = "silver_economia"
GOLD   = "gold_economia"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")

print("✅ Schemas configurados:", BRONZE, SILVER, GOLD)

In [ ]:
# ==========================================================
# 2. INGESTÃO BRONZE: IPCA (API Banco Central) e Boi Gordo
# ==========================================================
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp

# Ingestão IPCA via API REST do Banco Central
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
params = {"formato": "json", "dataInicial": "01/01/2024", "dataFinal": "31/12/2024"}

df_ipca_pd = pd.DataFrame(requests.get(url, params=params).json())
df_ipca_pd.columns = ["data", "ipca"]
df_ipca_pd["ipca"] = df_ipca_pd["ipca"].str.replace(",", ".").astype(float)

df_ipca_spark = spark.createDataFrame(df_ipca_pd).withColumn("data_coleta", current_timestamp())
df_ipca_spark.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE}.ipca")

# Ingestão Boi Gordo
boi_sample = [
    {"Data": "01/2024", "Valor": 248.50}, {"Data": "02/2024", "Valor": 242.10},
    {"Data": "03/2024", "Valor": 235.80}, {"Data": "04/2024", "Valor": 230.20},
    {"Data": "05/2024", "Valor": 225.40}, {"Data": "06/2024", "Valor": 228.90},
    {"Data": "07/2024", "Valor": 234.10}, {"Data": "08/2024", "Valor": 241.60},
    {"Data": "09/2024", "Valor": 258.30}, {"Data": "10/2024", "Valor": 304.50},
    {"Data": "11/2024", "Valor": 335.20}, {"Data": "12/2024", "Valor": 322.80}
]
df_boi_spark = spark.createDataFrame(pd.DataFrame(boi_sample)).withColumn("data_coleta", current_timestamp())
df_boi_spark.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE}.boi_gordo")

print("✅ Camada Bronze populada com sucesso!")

In [ ]:
# ==========================================================
# 3. CAMADA SILVER: Limpeza, Tipagem Estrita e Join Temporal
# ==========================================================
from pyspark.sql.functions import to_date, date_format, col, regexp_replace

ipca = spark.table(f"{CATALOG}.{BRONZE}.ipca")
boi  = spark.table(f"{CATALOG}.{BRONZE}.boi_gordo").withColumnRenamed("Data", "data").withColumnRenamed("Valor", "boi_gordo")

# Padronização para formato canônico 'yyyy-MM'
boi = boi.withColumn("data", date_format(to_date("data", "MM/yyyy"), "yyyy-MM"))
ipca = ipca.withColumn("data", date_format(to_date("data", "dd/MM/yyyy"), "yyyy-MM"))

# Inner Join por mês
ip = ipca.alias("ip")
bo = boi.alias("bo")

df_silver = (
    ip.join(bo, col("ip.data") == col("bo.data"), "inner")
      .select(
          to_date(col("ip.data"), "yyyy-MM").alias("data"),
          regexp_replace(col("ip.ipca"), ",", ".").cast("double").alias("ipca"),
          regexp_replace(col("bo.boi_gordo"), ",", ".").cast("double").alias("boi_gordo"),
          col("ip.data_coleta").alias("data_coleta")
      )
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SILVER}.economia")
display(df_silver)

In [ ]:
# ==========================================================
# 4. CAMADA GOLD: Window Functions e Métricas de Negócio
# ==========================================================
from pyspark.sql import functions as F, Window

df_silver = spark.table(f"{CATALOG}.{SILVER}.economia")
w = Window.orderBy("data")

df_gold = (
    df_silver
    .withColumn("ipca_ant", F.lag("ipca").over(w))
    .withColumn("boi_ant", F.lag("boi_gordo").over(w))
    .withColumn("variacao_ipca", F.round((F.col("ipca") - F.col("ipca_ant")) / F.col("ipca_ant") * 100, 2))
    .withColumn("variacao_boi", F.round((F.col("boi_gordo") - F.col("boi_ant")) / F.col("boi_ant") * 100, 2))
    .drop("ipca_ant", "boi_ant")
)

df_gold.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.insights")
display(df_gold)

In [ ]:
# ==========================================================
# 5. VIEW ANALÍTICA DE CONSUMO (Gold Dashboard)
# ==========================================================
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD}.vw_gold_dashboard AS
SELECT
    data,
    ipca,
    boi_gordo,
    variacao_ipca,
    variacao_boi,
    ROUND((variacao_ipca + variacao_boi) / 2, 2) AS media_variacoes,
    CASE
        WHEN variacao_boi > variacao_ipca THEN 'Preço do boi cresce mais'
        WHEN variacao_ipca > variacao_boi THEN 'Inflação cresce mais'
        ELSE 'Empate'
    END AS destaque,
    CASE
        WHEN ABS(variacao_boi - variacao_ipca) > 5 THEN 'Alta divergência'
        WHEN ABS(variacao_boi - variacao_ipca) BETWEEN 2 AND 5 THEN 'Média divergência'
        ELSE 'Baixa divergência'
    END AS classe_impacto
FROM {CATALOG}.{GOLD}.insights;
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD}.vw_gold_dashboard ORDER BY data"))